In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# LightGBM Hierarchical Stacking Pipeline: Comparative Analysis of Baseline vs KFDA-Transformed Input Manifolds (`models/train_oof_logistic_regression_stacking_new.ipynb`)

This notebook trains and benchmarks the **LightGBM Hierarchical Stacking Pipeline natively in Python**, directly comparing:
1. **Previous Method (Baseline Stacking)**: Hierarchical LightGBM sub-models (`L1`, `L2`, `L3A`, `L3B`) trained on **38 Raw Scaled Features in $\mathbb{R}^{38}$** $\to$ Multinomial Logistic Regression meta-learner.
2. **KFDA Method (KFDA Stacking)**: Features mapped non-linearly via **Nystroem RBF Kernel approximation ($\mathbb{R}^{38} \to \mathbb{R}^{600}$)** and **Kernel Fisher Discriminant Analysis (KFDA / Multi-Class LDA) ($\mathbb{R}^{600} \to \mathbb{R}^{4}$)** $\to$ Hierarchical LightGBM / Balanced sub-models $\to$ Multinomial Logistic Regression meta-learner.

```mermaid
flowchart TD
    Data["Clean Complete Cases (5v_cleandf.RData)"] --> Split["3-Way Stratified Split: Train (98%), Val (1%), Test (1%)"]
    Split --> Feat["38 Engineered Clinical Features & Continuous Z-Score Scaling"]
    
    Feat --> PipeA["Method 1: Baseline Stacking (Raw 38 Features in R^38)"]
    PipeA --> L1A["Layer 1 LightGBM (ESI 1 vs 2..5 with SMOTE)"]
    PipeA --> L2A["Layer 2 LightGBM (ESI 2,3 vs 4,5 with SMOTE)"]
    PipeA --> L3AA["Layer 3A LightGBM (ESI 2 vs 3 with SMOTE)"]
    PipeA --> L3BA["Layer 3B LightGBM (ESI 4 vs 5 with SMOTE)"]
    L1A & L2A & L3AA & L3BA --> MetaA["Baseline Meta-Learner (LogisticRegression on Val Probs)"]
    
    Feat --> PipeB["Method 2: KFDA Stacking (Canonical Space in R^4)"]
    PipeB --> Nyst["Nystroem RBF Kernel Mapping phi(X) in R^600"]
    Nyst --> KFDA["KFDA / Multi-Class LDA Projection to R^4 Coordinates"]
    KFDA --> L1B["KFDA Layer 1 LightGBM with SMOTE"]
    KFDA --> L2B["KFDA Layer 2 LightGBM with SMOTE"]
    KFDA --> L3AB["KFDA Layer 3A LightGBM with SMOTE"]
    KFDA --> L3BB["KFDA Layer 3B LightGBM with SMOTE"]
    L1B & L2B & L3AB & L3BB --> MetaB["KFDA Meta-Learner (LogisticRegression on Val Probs)"]
    
    MetaA & MetaB --> Benchmark["Holdout Test Benchmark Comparison: Metrics, Confusion Matrices & ROC Curves"]
```

### 🔬 Benchmark Comparison Matrix
| Pipeline | Input Features | Sub-Model Training | Meta-Learner Calibration | Holdout Evaluation |
|---|---|---|---|---|
| **Baseline Stacking** | $\mathbb{R}^{38}$ (Raw Scaled) | Hierarchical LightGBM + SMOTE | 5-Fold CV on Validation Set (1%) | Test Set (1% complete cases) |
| **KFDA Stacking** | $\mathbb{R}^{4}$ (KFDA Coordinates) | Hierarchical LightGBM + SMOTE | 5-Fold CV on Validation Set (1%) | Test Set (1% complete cases) |

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:15],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Engineering (38 Features) & Continuous Scaling
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from rpy2.robjects import r
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.kernel_approximation import Nystroem
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, confusion_matrix
)
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :15]
y_train     = train_mat_in[:, 15].astype(int)

raw_mat_val = val_mat_in[:, :15]
y_val       = val_mat_in[:, 15].astype(int)

raw_mat_ts  = test_mat_in[:, :15]
y_test      = test_mat_in[:, 15].astype(int)

def build_38_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 38), dtype=np.float64)
    X[:, :15] = raw_mat
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]; resp_min = raw_mat[:, 8]; spo2_min = raw_mat[:, 9]; sbp_min = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]; resp_max = raw_mat[:, 12]; spo2_max = raw_mat[:, 13]; sbp_max = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X[:, 15] = (t_o2 < 90).astype(float)
    X[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X[:, 17] = (t_rr < 10).astype(float)
    X[:, 18] = (t_rr > 30).astype(float)
    X[:, 19] = (t_sbp <= 90).astype(float)
    X[:, 20] = (t_sbp > 220).astype(float)
    X[:, 21] = (t_hr < 40).astype(float)
    X[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X[:, 23] = (t_hr > 150).astype(float)
    X[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X[:, 25] = hr_rng; X[:, 26] = rr_rng; X[:, 27] = spo2_rng; X[:, 28] = sbp_rng
    X[:, 29] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max)
    X[:, 36] = hr_rng / (t_hr + 1.0)
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0
    return X

X_train_raw = build_38_feature_matrix(raw_mat_tr)
X_val_raw   = build_38_feature_matrix(raw_mat_val)
X_test_raw  = build_38_feature_matrix(raw_mat_ts)

feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]

cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_val[:, cont_cols_idx]   = scaler.transform(X_val_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])

print(f"Data Matrices Ready:")
print(f"  * Train Set : {X_train.shape[0]:,} visits x {X_train.shape[1]} features")
print(f"  * Val Set   : {X_val.shape[0]:,} visits")
print(f"  * Test Set  : {X_test.shape[0]:,} visits")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Pipeline A — Baseline Hierarchical LightGBM Stacking (Raw 38 Features in R^38)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE A: BASELINE LIGHTGBM STACKING (RAW 38 FEATURES IN R^38)")
print("=" * 80)

def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()

# Layer 1: ESI 1 vs (ESI 2..5)
X_sm1, y_sm1 = numpy_smote(X_train, (y_train == 1).astype(int))
l1_base = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_base.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 2: ESI 2,3 vs ESI 4,5 (trained on non-ESI 1)
m2_tr  = (y_train != 1); m2_val = (y_val != 1)
X_sm2, y_sm2 = numpy_smote(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_base = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_base.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3A: ESI 2 vs ESI 3
m3a_tr  = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
X_sm3a, y_sm3a = numpy_smote(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_base = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_base.fit(X_sm3a, y_sm3a, eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3B: ESI 4 vs ESI 5
m3b_tr  = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
X_sm3b, y_sm3b = numpy_smote(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_base = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_base.fit(X_sm3b, y_sm3b, eval_set=[(X_val[m3b_val], np.isin(y_val[m3b_val], [4, 5]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Baseline Hierarchical LightGBM sub-models trained in {time.time()-t0:.1f}s!")

# Generate Validation Probabilities for Meta-Learner
def compute_hierarchical_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

val_probs_base  = compute_hierarchical_probs(l1_base, l2_base, l3a_base, l3b_base, X_val)
test_probs_base = compute_hierarchical_probs(l1_base, l2_base, l3a_base, l3b_base, X_test)

# Fit Baseline Multinomial Logistic Regression Meta-Learner
meta_base = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_base.fit(val_probs_base, y_val)

preds_base = meta_base.predict(test_probs_base)
probs_base = meta_base.predict_proba(test_probs_base)
print("✓ Baseline Stacking Pipeline Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Train Pipeline B — KFDA-Transformed Inputs + Hierarchical Stacking (R^4)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE B: KFDA-TRANSFORMED INPUTS + STACKING (R^4 MANIFOLD)")
print("=" * 80)

# 1. Nystroem Non-Linear RBF Landmark Mapping (R^38 -> R^600)
nystroem_kfda = Nystroem(
    kernel='rbf',
    gamma=0.15,
    n_components=600,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
Phi_train = nystroem_kfda.fit_transform(X_train)
Phi_val   = nystroem_kfda.transform(X_val)
Phi_test  = nystroem_kfda.transform(X_test)
print(f"✓ Nystroem Kernel Mapping fitted in {time.time()-t0:.1f}s (Shape: {Phi_train.shape})")

# 2. Multi-Class Linear Discriminant Analysis on Kernel Space (KFDA Projection to R^4)
kfda_lda = LinearDiscriminantAnalysis(n_components=4)
Z_train  = kfda_lda.fit_transform(Phi_train, y_train)
Z_val    = kfda_lda.transform(Phi_val)
Z_test   = kfda_lda.transform(Phi_test)
print(f"✓ KFDA Fisher Projection fitted in {time.time()-t0:.1f}s (Explained Variance: {np.round(kfda_lda.explained_variance_ratio_, 4)})")

# 3. Hierarchical LightGBM sub-models trained on the KFDA Manifold (Z in R^4)
t0 = time.time()

# KFDA Layer 1: ESI 1 vs (ESI 2..5)
Z_sm1, y_kfda_sm1 = numpy_smote(Z_train, (y_train == 1).astype(int))
l1_kfda = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_kfda.fit(Z_sm1, y_kfda_sm1, eval_set=[(Z_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# KFDA Layer 2: ESI 2,3 vs ESI 4,5
Z_sm2, y_kfda_sm2 = numpy_smote(Z_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_kfda = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_kfda.fit(Z_sm2, y_kfda_sm2, eval_set=[(Z_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# KFDA Layer 3A: ESI 2 vs ESI 3
Z_sm3a, y_kfda_sm3a = numpy_smote(Z_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_kfda = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_kfda.fit(Z_sm3a, y_kfda_sm3a, eval_set=[(Z_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# KFDA Layer 3B: ESI 4 vs ESI 5
Z_sm3b, y_kfda_sm3b = numpy_smote(Z_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_kfda = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_kfda.fit(Z_sm3b, y_kfda_sm3b, eval_set=[(Z_val[m3b_val], np.isin(y_val[m3b_val], [4, 5]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ KFDA Hierarchical LightGBM sub-models trained in {time.time()-t0:.1f}s!")

val_probs_kfda  = compute_hierarchical_probs(l1_kfda, l2_kfda, l3a_kfda, l3b_kfda, Z_val)
test_probs_kfda = compute_hierarchical_probs(l1_kfda, l2_kfda, l3a_kfda, l3b_kfda, Z_test)

# Fit KFDA Multinomial Logistic Regression Meta-Learner
meta_kfda = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_kfda.fit(val_probs_kfda, y_val)

preds_kfda = meta_kfda.predict(test_probs_kfda)
probs_kfda = meta_kfda.predict_proba(test_probs_kfda)
print("✓ KFDA Stacking Pipeline Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Side-by-Side Holdout Test Benchmark Evaluation (Baseline vs KFDA)
# ---------------------------------------------------------
y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]

def eval_pipeline(y_true, y_pred, y_prob, name):
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mf1     = f1_score(y_true, y_pred, average='macro', zero_division=0)
    wf1     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    auc_ovr = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
    
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    rec_per = recall_score(y_true, y_pred, average=None, zero_division=0)
    prec_per = precision_score(y_true, y_pred, average=None, zero_division=0)
    f1_per   = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    spec_per = []
    for c in range(5):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        spec_per.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    
    auc_per = [roc_auc_score(y_test_bin[:, c], y_prob[:, c]) for c in range(5)]
    
    return {
        'name': name,
        'acc': acc, 'bal_acc': bal_acc, 'macro_spec': np.mean(spec_per),
        'macro_f1': mf1, 'weighted_f1': wf1, 'auc_ovr': auc_ovr,
        'rec_per': rec_per, 'spec_per': spec_per, 'prec_per': prec_per,
        'f1_per': f1_per, 'auc_per': auc_per, 'cm': cm
    }

res_base = eval_pipeline(y_test, preds_base, probs_base, 'Baseline Stacking (Raw R^38)')
res_kfda = eval_pipeline(y_test, preds_kfda, probs_kfda, 'KFDA Stacking (R^4 Manifold)')

# Overall Benchmark Summary Table
overall_comp_df = pd.DataFrame([
    {
        'Metric': 'Overall Accuracy',
        'Baseline Stacking (R^38)': f"{res_base['acc']*100:.2f}%",
        'KFDA Stacking (R^4)': f"{res_kfda['acc']*100:.2f}%",
        'Delta (KFDA - Base)': f"{(res_kfda['acc'] - res_base['acc'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro Balanced Accuracy',
        'Baseline Stacking (R^38)': f"{res_base['bal_acc']*100:.2f}%",
        'KFDA Stacking (R^4)': f"{res_kfda['bal_acc']*100:.2f}%",
        'Delta (KFDA - Base)': f"{(res_kfda['bal_acc'] - res_base['bal_acc'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro Specificity (TNR)',
        'Baseline Stacking (R^38)': f"{res_base['macro_spec']*100:.2f}%",
        'KFDA Stacking (R^4)': f"{res_kfda['macro_spec']*100:.2f}%",
        'Delta (KFDA - Base)': f"{(res_kfda['macro_spec'] - res_base['macro_spec'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro ROC-AUC (OvR)',
        'Baseline Stacking (R^38)': f"{res_base['auc_ovr']:.4f}",
        'KFDA Stacking (R^4)': f"{res_kfda['auc_ovr']:.4f}",
        'Delta (KFDA - Base)': f"{res_kfda['auc_ovr'] - res_base['auc_ovr']:+.4f}"
    },
    {
        'Metric': 'Macro F1-Score',
        'Baseline Stacking (R^38)': f"{res_base['macro_f1']:.4f}",
        'KFDA Stacking (R^4)': f"{res_kfda['macro_f1']:.4f}",
        'Delta (KFDA - Base)': f"{res_kfda['macro_f1'] - res_base['macro_f1']:+.4f}"
    },
    {
        'Metric': 'Weighted F1-Score',
        'Baseline Stacking (R^38)': f"{res_base['weighted_f1']:.4f}",
        'KFDA Stacking (R^4)': f"{res_kfda['weighted_f1']:.4f}",
        'Delta (KFDA - Base)': f"{res_kfda['weighted_f1'] - res_base['weighted_f1']:+.4f}"
    }
])

print("=" * 95)
print("   HOLDOUT TEST OVERALL BENCHMARK: BASELINE STACKING vs KFDA STACKING")
print("=" * 95)
print(overall_comp_df.to_string(index=False))
print("=" * 95 + chr(10))

# Per-Class Detailed Breakdown Table
per_class_rows = []
for c in range(5):
    cls_name = f'ESI {c+1}'
    per_class_rows.append({
        'Triage_Level': cls_name,
        'True_Count': int(np.sum(y_test == (c+1))),
        'Base_Recall': f"{res_base['rec_per'][c]*100:.2f}%",
        'KFDA_Recall': f"{res_kfda['rec_per'][c]*100:.2f}%",
        'Delta_Recall': f"{(res_kfda['rec_per'][c] - res_base['rec_per'][c])*100:+.2f}%",
        'Base_Spec': f"{res_base['spec_per'][c]*100:.2f}%",
        'KFDA_Spec': f"{res_kfda['spec_per'][c]*100:.2f}%",
        'Base_ROC_AUC': round(res_base['auc_per'][c], 4),
        'KFDA_ROC_AUC': round(res_kfda['auc_per'][c], 4),
        'Delta_AUC': round(res_kfda['auc_per'][c] - res_base['auc_per'][c], 4)
    })

per_class_df = pd.DataFrame(per_class_rows)
print("PER-CLASS RECALL & ROC-AUC COMPARISON:")
print(per_class_df.to_string(index=False))
print("=" * 95 + chr(10))

# Export Comparison CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'oof_stacking_baseline_vs_kfda_report.csv')
per_class_df.to_csv(report_file, index=False)
print(f"✓ Benchmark comparison report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Side-by-Side 5x5 Confusion Matrix Comparison (Baseline vs KFDA)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
esi_labels = [f"ESI {i}" for i in range(1, 6)]

# Left: Baseline Stacking
cm_base = res_base['cm']
cm_base_norm = cm_base.astype('float') / cm_base.sum(axis=1)[:, np.newaxis]
annot_base = np.empty_like(cm_base, dtype=object)
for i in range(5):
    for j in range(5):
        annot_base[i, j] = f"{cm_base[i, j]:,}\n({cm_base_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_base_norm, annot=annot_base, fmt='', cmap='Blues', cbar=True, ax=axes[0],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[0].set_title(
    f"Method 1: Baseline Stacking (Raw Features in R^38)\n"
    f"Balanced Acc: {res_base['bal_acc']*100:.2f}% | Macro ROC-AUC: {res_base['auc_ovr']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

# Right: KFDA Stacking
cm_kfda = res_kfda['cm']
cm_kfda_norm = cm_kfda.astype('float') / cm_kfda.sum(axis=1)[:, np.newaxis]
annot_kfda = np.empty_like(cm_kfda, dtype=object)
for i in range(5):
    for j in range(5):
        annot_kfda[i, j] = f"{cm_kfda[i, j]:,}\n({cm_kfda_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_kfda_norm, annot=annot_kfda, fmt='', cmap='Greens', cbar=True, ax=axes[1],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[1].set_title(
    f"Method 2: KFDA Stacking (Canonical Space R^4)\n"
    f"Balanced Acc: {res_kfda['bal_acc']*100:.2f}% | Macro ROC-AUC: {res_kfda['auc_ovr']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[1].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_comp_path = os.path.join(plots_dir, "oof_stacking_side_by_side_confusion_matrix.png")
plt.savefig(cm_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_side_by_side_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side Confusion Matrix comparison saved to: {cm_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Side-by-Side Multiclass ROC-AUC Curves (Baseline vs KFDA)
# ---------------------------------------------------------
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

def plot_roc_curves_on_ax(ax, probs, title_text):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i in range(5):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    # Micro average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    # Macro average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(5)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(5):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= 5
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_curves_on_ax(axes[0], probs_base, f"Method 1: Baseline Stacking ROC Curves (Macro AUC = {res_base['auc_ovr']:.4f})")
plot_roc_curves_on_ax(axes[1], probs_kfda, f"Method 2: KFDA Stacking ROC Curves (Macro AUC = {res_kfda['auc_ovr']:.4f})")

plt.tight_layout()
roc_comp_path = os.path.join(plots_dir, "oof_stacking_kfda_vs_baseline_roc_auc.png")
plt.savefig(roc_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_kfda_vs_baseline_roc_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side ROC Curves saved to: {roc_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Comparison Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names,
    'baseline_pipeline': {
        'l1_base': l1_base,
        'l2_base': l2_base,
        'l3a_base': l3a_base,
        'l3b_base': l3b_base,
        'meta_base': meta_base
    },
    'kfda_pipeline': {
        'nystroem_kfda': nystroem_kfda,
        'kfda_lda': kfda_lda,
        'l1_kfda': l1_kfda,
        'l2_kfda': l2_kfda,
        'l3a_kfda': l3a_kfda,
        'l3b_kfda': l3b_kfda,
        'meta_kfda': meta_kfda
    }
}

bundle_file = os.path.join(deploy_dir, 'oof_stacking_baseline_vs_kfda_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='OOF_Stacking_Baseline_vs_KFDA_Benchmark',
    dataset='datasets/5v_cleandf.RData',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    holdout_test_samples=len(y_test),
    baseline_metrics=dict(
        overall_accuracy=round(res_base['acc'], 4),
        macro_balanced_accuracy=round(res_base['bal_acc'], 4),
        macro_specificity=round(res_base['macro_spec'], 4),
        macro_roc_auc_ovr=round(res_base['auc_ovr'], 4),
        macro_f1=round(res_base['macro_f1'], 4)
    ),
    kfda_metrics=dict(
        overall_accuracy=round(res_kfda['acc'], 4),
        macro_balanced_accuracy=round(res_kfda['bal_acc'], 4),
        macro_specificity=round(res_kfda['macro_spec'], 4),
        macro_roc_auc_ovr=round(res_kfda['auc_ovr'], 4),
        macro_f1=round(res_kfda['macro_f1'], 4)
    ),
    per_class_comparison=per_class_rows
)

manifest_file = os.path.join(deploy_dir, 'oof_stacking_baseline_vs_kfda_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Comparison Bundle   : {bundle_file}")
print(f"✓ Comparison Manifest : {manifest_file}")